1) Create a data frame that contains the following information for each drug: the unique DrugBank identifier, the drug's name, its type, description, dosage form, indications, mechanism of action, and information on food interactions.

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

file_path = 'drugbank_partial.xml'
namespaces = {'drugbank': 'http://www.drugbank.ca'}

data = []
depth = 0

# Parse the XML file and extract relevant information
for event, elem in ET.iterparse(file_path, events=('start', 'end')):
    if event == 'start':
        depth += 1
    elif event == 'end':
        depth -= 1

    # Process each drug element when the end tag is encountered
    if elem.tag == f"{{{namespaces['drugbank']}}}drug" and event == 'end' and depth == 1:
        drug_data = {}

        # Extract the primary drugbank-id
        drugbank_id_elem = elem.find('drugbank:drugbank-id[@primary="true"]', namespaces)
        if drugbank_id_elem is not None:
            drug_data['drugbank-id'] = drugbank_id_elem.text

        # Extract other fields like name, description, state, indication, and mechanism-of-action
        for field in ['name', 'description', 'state', 'indication', 'mechanism-of-action']:
            field_elem = elem.find(f'drugbank:{field}', namespaces)
            if field_elem is not None:
                drug_data[field] = field_elem.text

        # Extract food interactions
        drug_data['food-interactions'] = []
        for food_interaction in elem.findall('.//drugbank:food-interaction', namespaces):
            drug_data['food-interactions'].append(food_interaction.text)

        data.append(drug_data)
        elem.clear()

# Create a DataFrame from the extracted data
df = pd.DataFrame(data)
df

,drugbank-id,name,description,state,indication,mechanism-of-action,food-interactions
0,DB00001,Lepirudin,Lepirudin is a recombinant hirudin formed by 6...,solid,Lepirudin is indicated for anticoagulation in ...,Lepirudin is a direct thrombin inhibitor used ...,[Avoid herbs and supplements with anticoagulan...
1,DB00002,Cetuximab,Cetuximab is a recombinant chimeric human/mous...,liquid,Cetuximab indicated for the treatment of local...,The epidermal growth factor receptor (EGFR) is...,[]
2,DB00003,Dornase alfa,Dornase alfa is a biosynthetic form of human d...,liquid,Used as adjunct therapy in the treatment of cy...,Dornase alfa is a biosynthetic form of human D...,[]
3,DB00004,Denileukin diftitox,A recombinant DNA-derived cytotoxic protein co...,liquid,For treatment of cutaneous T-cell lymphoma,Denileukin diftitox binds to the high-affinity...,[]
4,DB00005,Etanercept,Dimeric fusion protein consisting of the extra...,liquid,Etanercept is indicated for the treatment of m...,There are two distinct receptors for TNF (TNFR...,[]
...,...,...,...,...,...,...,...
95,DB00104,Octreotide,Acromegaly is a disorder caused by excess grow...,solid,Octreotide by injection is used for the treatm...,Octreotide binds to somatostatin receptors cou...,[Take on an empty stomach. The oral capsules s...
96,DB00105,Interferon alfa-2b,Interferon alpha 2b (human leukocyte clone hif...,liquid,"For the treatment of hairy cell leukemia, mali...",Interferon alpha binds to type I interferon re...,[Avoid alcohol.]
97,DB00106,Abarelix,Synthetic decapeptide antagonist to gonadotrop...,solid,For palliative treatment of advanced prostate ...,Abarelix binds to the gonadotropin releasing h...,[]
98,DB00107,Oxytocin,Sir Henry H. Dale first identified oxytocin an...,liquid,Administration of exogenous oxytocin is indica...,Oxytocin plays a vital role in labour and deli...,[]
